In [2]:
# Входные данные по задаче
n = 2 # Количество параметров в формуле ранжирования
a = [1, 100] # Целые числа
d = 10 # Количество объектов для ранжирования
f = [[1, 2], [2, 1], [3, 1],
     [4, 1], [5, 1], [6, 1], 
     [7, 1], [8, 1], [9, 1],
     [10, 1]] # Числовые признаки i-го объекта
q = 4 # Количество запросов к системе ранжирования
k = [[1, 2], [1, 10], [2, 4, 1, 1000], [1, 10]] # Запросы

In [ ]:
# Значение любого родительского узла больше или равно значениям его потомков. 
# В корне всегда находится максимальный элемент
class Tree:
    def __init__(self): # Инициализация
        self.heap = []
        self.positions = {}

    def build(self, docs): # Эта функция нужна для создания дерева из листа с данными о релевантности объекта О(d)
        self.heap = docs # Переносим данные из листа с релевантностью и номером документа
        self.positions = {self.heap[i][1]: i for i in range(len(self.heap))} # Тут создается словарь позиций, где каждому номеру документа соответствует его нынешняя позиция в дереве
        for i in range(len(self.heap) // 2 - 1, -1, -1): # А здесь с конца дерева (последнего узла с листьями) начинаем ее строить
            self.down(i)

    def swap(self, i, j): # Эта функция нужна для сортировки (просто меняем позиции местами) О(1)
        self.heap[i], self.heap[j] = self.heap[j], self.heap[i]
        self.positions[self.heap[i][1]] = i
        self.positions[self.heap[j][1]] = j

    def up(self, i): # Это для поднятися узлов наверх O(log d)
        while i > 0:
            parent = (i - 1) // 2 # Ищем индекс родительского узла
            if self.heap[i][0] > self.heap[parent][0]: # Условие на то, чтобы заменить родителя его узлом
                self.swap(i, parent)
                i = parent # Чтобы идти дальше вверх по дереву, так как теперь позиция нашего изначального узла == позиции его родителя
            else:
                break
        
    def down(self, i): # Это чтобы опустить вниз O(log d)
        while True:
            left = 2 * i + 1
            right = 2 * i + 2
            big = i
            # Нужно найти узел, который будет больше, чем тот, что мы подали в функции
            if left < len(self.heap):
                if self.heap[left][0] > self.heap[big][0]:
                    big = left
            if right < len(self.heap):
                if self.heap[right][0] > self.heap[big][0]:
                    big = right
            if big == i:
                break
            self.swap(i, big)
            i = big # И идем дальше вниз по дереву

    def update(self, object_id, new_rating): # Обновить рейтинг по запросу пользователя O(log d) за счет передвижений объекта
        i = self.positions[object_id] # Определяем позицию в дереве
        old_rating = self.heap[i][0] # Достаем старый рейтинг из дерева
        self.heap[i] = (new_rating, object_id) # Обновляем данные внутри дерева
        # Тут двигаем вверх или вниз в зависимости от того вырос рейтинг или упал
        if new_rating > old_rating:
            self.up(i)
        elif new_rating < old_rating:
            self.down(i)

    def relev_search(self, k): # Поиск по запросу пользователя
        if not self.heap: # В пустом не ищем
            return []
        results = []
        candidates = [0] # Самый максимальный рейтинг будет у корня
        for _ in range(k): # Здесь собираем результаты и добавляем листья в кандидаты
            max_id = 0 
            for i in range(1, len(candidates)):
                if self.heap[candidates[i]][0] > self.heap[candidates[max_id]][0]: # Сравниваем значение кандидата с максимальным имеющимся
                    max_id = i 
            heap_index = candidates.pop(max_id) # Сохраняем индекс, но убираем из кандидатов, так как перекидываем в результаты
            object_id = self.heap[heap_index][1] # Сохраняем номер документа
            results.append(object_id) # Добавляем номер документа в результаты
            left = 2 * heap_index + 1
            if left < len(self.heap): # Сохраняем листья в кандидаты
                candidates.append(left)
            right = 2 * heap_index + 2
            if right < len(self.heap):
                candidates.append(right)
        return results


In [ ]:
docs = []
for i in range(d): # В этом цикле собираем рейтинги в лист O(nd)
    rating = 0
    for j in range(n):
        rating += a[j] * f[i][j]
    docs.append([rating, i + 1])

print(docs)

[[201, 1], [102, 2], [103, 3], [104, 4], [105, 5], [106, 6], [107, 7], [108, 8], [109, 9], [110, 10]]


In [ ]:
rel = Tree()
rel.build(docs)

for i in range(q): # В этом цикле обработка запросов по условию задачи
    if k[i][0] == 1:
        print(*rel.relev_search(k[i][1]))
    elif k[i][0] == 2:
        d = k[i][1]
        j = k[i][2]
        v = k[i][3]
        rel_id = rel.positions[d] # Позиция в куче
        old_rating = rel.heap[rel_id][0]
        new_rating = (old_rating - a[j - 1] * f[d - 1][j - 1] + a[j - 1] * v) # Вычитаю старый вклад признака из старого рейтинга и добавляю новый вклад признака
        f[d - 1][j - 1] = v # Обновляем полученные значения признаков
        rel.update(d, new_rating) # Передаю номер документа, внутри функция сама достанет позицию
    

1 10
1 10 9 8 7 6 5 4 3 2
4 1 10 9 8 7 6 5 3 2


У вас есть данные о ценах акций за n дней в виде массива prices. Вы можете совершить не более k сделок (покупка + продажа = 1 сделка).
Найдите максимальную прибыль, которую можно получить, соблюдая правила:

–	Нельзя совершать несколько сделок одновременно (нельзя купить и продать в один и тот же день)

–	Нельзя покупать, если уже есть купленные акции

–	Можно совершать любое количество сделок, но не более k

Пример:
prices = [3, 2, 6, 5, 0, 3], k = 2
Максимальная прибыль: 7
Объяснение: 
Покупка в день 2 (цена 2), продажа в день 3 (цена 6) → прибыль 4
Покупка в день 5 (цена 0), продажа в день 6 (цена 3) → прибыль 3
Итого: 4 + 3 = 7


In [1]:
prices = [3, 2, 6, 5, 0, 3]
k = 2

In [2]:
# Временная сложность O(nk) потому что мы проходим весь массив prices k раз

sell = [] # Массив для отслеживания выгоды продажи акций
pricing = [] # Массив для отслеживания выгоды покупки акций
for i in range(k):
    # Строка - номер сделки, столбец - номер дня
    sell.append([float('-inf')] * len(prices))
    pricing.append([float('-inf')] * len(prices))
for i in range(k):
    for j in range(len(prices)):
        if j == 0:
            if i == 0:
                pricing[i][j] = -prices[j] # Тут минусуем стоимость акции первой сделки первого дня, так как у нас нет акции и мы можем ее только купить без сравнения с предыдущими днями
            continue
        if i == 0:
            pricing[i][j] = max(pricing[i][j-1], -prices[j]) # Первая сделка учитывает только выгоду от покупки без полученного результата предыдущих сделок
        else:
            pricing[i][j] = max(pricing[i][j-1], sell[i-1][j-1] - prices[j])  # А здесь уже учитываем сколько денег у нас уже есть с продажи
        sell[i][j] = max(sell[i][j-1], pricing[i][j-1] + prices[j]) # Тут выбираем между удерживанием и продажей сегодня

print(max(0, *(row[-1] for row in sell)))

7


Дана информация о времени заезда и отъезда посетителей отеля. Необходимо определить, в какой день посетителей в отеле единомоментно находилось больше всего.

Пример входных данных (один элемент данного листа – кортеж, содержащий дату заезда и отъезда одного посетителя): [(“2024-09-15”, “2024-09-15”), (“2024-09-14”, “2024-09-21”)]

In [17]:
guests = [('2024-09-15', '2024-09-15'), ('2024-09-12', '2024-09-21'), ('2024-09-17', '2024-09-21')]

In [ ]:
arrivals = {} # Все [0] в кортежах: количество человек
departures = {} # Все [1] в кортежах: количество человек

for i in range(len(guests)): # Тут один раз прохожу по изначальному списку целиком O(n)?
    arrivals[guests[i][0]] = arrivals.get(guests[i][0], 0) + 1
    departures[guests[i][1]] = departures.get(guests[i][1], 0) - 1

all_dates = sorted(set(arrivals) | set(departures)) # Тут надо менять на функцию сортировки! O(nlog n) по идее на основной питоновской сортировке
current_guests = 0 
max_guests = 0
max_date = ''

for date in all_dates: # Тут прохожусь по всем датам, которые у меня есть
    current_guests += arrivals.get(date, 0)
    if current_guests > max_guests: # Проверка до отъезда, чтобы учитывать гостей, которые приехали и уехали одним днем
            max_guests = current_guests
            max_date = date
    current_guests += departures.get(date, 0) 
# У цикла тоже О(n) по идее?
    
print(max_date)
         

2024-09-15
